# Snowflake 101 — Part 1

## Warehouses, transformations and finding things

You run every cell in this notebook yourself. Nothing here depends on anyone else's account, so you can work at your own pace and re-run anything.

The data is Fiserv acquiring data: 5,000 merchants and 500,000 card transactions from June to September 2025, in euro. It is synthetic, but it is shaped like the real thing, and it is the same business you will work on tomorrow.

By the end of this notebook you will have:

1. Measured what a virtual warehouse actually does to query time
2. Found out why the smaller warehouse sometimes wins
3. Put a spending limit on compute with a resource monitor
4. Built a transformation and saved it as a table
5. Located an object you did not know the name of

Allow about 50 minutes. Run the cells in order.

## 0. Provision your account

Your account is empty. You build it yourself, and it runs entirely server-side: no client,
no CLI, no local install. `EXECUTE IMMEDIATE FROM` reads the SQL straight off an internal
stage and executes it in your account.

Run the next three cells in order, then carry on. **Start the day 2 build now even though
you will not touch it until tomorrow** — it takes about six minutes and finishes while the
101 slides are running. Leaving it until tomorrow morning wastes six minutes of lab time.

In [ ]:
%%sql -r provision_101
-- Builds FISERV_101_DB for today's three sessions. Seconds, not minutes: the 101 data is
-- deliberately small at 5,000 merchants and 500,000 transactions.
EXECUTE IMMEDIATE FROM @FISERV_SETUP.PUBLIC.WORKSHOP/branches/main/labs/101/generators/00_setup_all_101.sql;

In [ ]:
%%sql -r provision_payments
-- Builds FISERV_PAYMENTS_DB for tomorrow. Roughly six minutes: 2,000,000 merchants,
-- 30.2 million fee lines, five dynamic tables, two Cortex Search services and a semantic
-- view. Kick it off now and ignore it; it finishes while the slides run.
--
-- It deliberately does NOT build the Cortex Agent or the evaluation set. Those are the
-- session 5 exercises, and pre-building them would hand you the answers.
EXECUTE IMMEDIATE FROM @FISERV_SETUP.PUBLIC.WORKSHOP/branches/main/labs/data-intelligence-app/generators/00_setup_all.sql;

In [ ]:
%%sql -r verify_101
-- Every row must say PASS. If any row says FAIL, tell your facilitator the CHECK_NAME
-- before trying to fix it. Do not start the lab on a broken account.
EXECUTE IMMEDIATE FROM @FISERV_SETUP.PUBLIC.WORKSHOP/branches/main/labs/101/generators/99_verify_101.sql;

Expect **11 PASS rows** across three result sets: nine data and schema checks, then the
warehouse check, then the role check.

The warehouse one matters more than it looks. Module 1 compares a cold X-Small against a
cold Small, so if `FISERV_101_BIG_WH` is missing the whole first exercise has no second
half to compare against.

In [ ]:
%%sql -r ctx
-- Pins your role, database, schema and warehouse for the rest of the notebook.
-- Re-run this cell if a later cell complains that it cannot find an object.
USE ROLE ACCOUNTADMIN;
USE DATABASE FISERV_101_DB;
USE SCHEMA RAW;
USE WAREHOUSE FISERV_101_WH;

## 1. What is in the data

Two tables. `MERCHANTS` is who you acquire for, `TRANSACTIONS` is what they took.

Start by looking at the merchants. A **merchant segment** is Fiserv's own sizing of the merchant — SMB, Mid-Market or Enterprise. An **acquiring region** is which of the four European books the merchant sits in.

In [ ]:
%%sql -r merchants_sample
-- First look at the merchant master. Ten rows is enough to see the shape.
SELECT MERCHANT_ID, MERCHANT_NAME, MERCHANT_CATEGORY, MERCHANT_SEGMENT,
       COUNTRY_CODE, ACQUIRING_REGION, TERMINAL_COUNT
FROM FISERV_101_DB.RAW.MERCHANTS
LIMIT 10;

In [ ]:
%%sql -r txn_shape
-- How much transaction data there is, and the window it covers.
SELECT COUNT(*) AS TRANSACTION_COUNT,
       MIN(TRANSACTION_TIMESTAMP) AS EARLIEST,
       MAX(TRANSACTION_TIMESTAMP) AS LATEST,
       ROUND(SUM(AMOUNT_EUR), 2) AS TOTAL_EUR
FROM FISERV_101_DB.RAW.TRANSACTIONS;

## 2. Virtual warehouses

A **virtual warehouse** is the compute that runs your query. It is separate from the data: the data sits in storage, and you point whatever amount of compute you want at it. Warehouses come in sizes, each step doubling the compute and doubling the cost per second.

That is the claim. You are about to test it.

First, turn off the **result cache**. Snowflake remembers the answer to a query it has already run and returns it instantly without using any compute. That is excellent in production and useless for a timing test, because the second run would take no time at all regardless of warehouse size.

In [ ]:
%%sql -r no_cache
-- Turn off the result cache for this session so timings reflect real work.
-- Leave it off for the rest of the notebook.
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

### The test query

A 500,000-row aggregate finishes too fast to measure, so the query below deliberately does more work: it joins every transaction to its merchant, then to every other merchant in the same region and category. That produces about 88 million intermediate rows before collapsing back to one.

It is not a query you would write on purpose. It is a query that takes long enough to time.

This runs on **X-Small** in roughly **0.4 seconds**. Note the number Snowflake reports
under the cell; you will compare three of them.

Note the warehouse in the next cell: `FISERV_101_TIMING_WH`, not the `FISERV_101_WH` you
have been using. It exists only for this exercise and has run nothing so far, which is
what makes run 1 a fair first run.

In [ ]:
%%sql -r run_xsmall_cold
-- Timed run 1 of 3: X-Small, genuinely cold. FISERV_101_TIMING_WH exists only for
-- this exercise and has not been used yet, so nothing is cached on it. The earlier
-- cells in this notebook ran on FISERV_101_WH, which is why we do not reuse it here.
USE WAREHOUSE FISERV_101_TIMING_WH;
SELECT COUNT(DISTINCT t.TRANSACTION_ID) AS DISTINCT_TRANSACTIONS
FROM FISERV_101_DB.RAW.TRANSACTIONS t
JOIN FISERV_101_DB.RAW.MERCHANTS m
  ON t.MERCHANT_ID = m.MERCHANT_ID
JOIN FISERV_101_DB.RAW.MERCHANTS m2
  ON m2.ACQUIRING_REGION = m.ACQUIRING_REGION
 AND m2.MERCHANT_CATEGORY = m.MERCHANT_CATEGORY;

### Same query, one size up

`FISERV_101_BIG_WH` is a Small warehouse: twice the compute of an X-Small, and twice the
cost per second.

The usual assumption is that double the compute halves the time, so the run costs the same
in total. Before you run it, write down what you expect. Then check whether it happens.

In [ ]:
%%sql -r run_small_cold
-- Timed run 2 of 3: Small warehouse, same query, still no caching in play.
USE WAREHOUSE FISERV_101_BIG_WH;
SELECT COUNT(DISTINCT t.TRANSACTION_ID) AS DISTINCT_TRANSACTIONS
FROM FISERV_101_DB.RAW.TRANSACTIONS t
JOIN FISERV_101_DB.RAW.MERCHANTS m
  ON t.MERCHANT_ID = m.MERCHANT_ID
JOIN FISERV_101_DB.RAW.MERCHANTS m2
  ON m2.ACQUIRING_REGION = m.ACQUIRING_REGION
 AND m2.MERCHANT_CATEGORY = m.MERCHANT_CATEGORY;

### Now go back to the small one

Run the identical query on the X-Small again. The result cache is off, so Snowflake has to
do the work again rather than hand back a stored answer.

It took about 0.4 seconds on this warehouse a moment ago. Predict the number before you run it.

In [ ]:
%%sql -r run_xsmall_warm
-- Timed run 3 of 3: back to the same X-Small as run 1, which still holds the data
-- it read locally. Same query, same warehouse, same size. Only the cache differs.
USE WAREHOUSE FISERV_101_TIMING_WH;
SELECT COUNT(DISTINCT t.TRANSACTION_ID) AS DISTINCT_TRANSACTIONS
FROM FISERV_101_DB.RAW.TRANSACTIONS t
JOIN FISERV_101_DB.RAW.MERCHANTS m
  ON t.MERCHANT_ID = m.MERCHANT_ID
JOIN FISERV_101_DB.RAW.MERCHANTS m2
  ON m2.ACQUIRING_REGION = m.ACQUIRING_REGION
 AND m2.MERCHANT_CATEGORY = m.MERCHANT_CATEGORY;

### What just happened

Two things, and the second is the one worth remembering.

**Doubling the warehouse barely helped.** The Small finished at roughly the same time as the
cold X-Small, near enough that the difference is inside normal run-to-run variation. You paid
twice the credits per second for it. Depending on the run, the Small may even come out
slightly slower; that is the point, not a mistake.

**Running it a second time roughly halved the time.** Same warehouse, same size, same query,
about half the duration.

The reason is the **local cache**. Each warehouse keeps the table data it has recently read on
its own SSD. Run 1 fetched from storage. Run 3 already had the data locally and skipped the
slowest part of the job. Run 2 on the Small was starting cold, exactly like run 1, which is why
the extra compute bought so little.

Two things follow, and both matter more than the size number:

- **Size affects compute, not data fetching.** When a query is dominated by reading data for
  the first time, a bigger warehouse helps far less than you expect. Sizing up is the right
  answer for queries that are genuinely compute-bound, especially ones spilling to disk.
- **A warm warehouse is a fast warehouse.** This is why aggressively suspending warehouses can
  backfire, and why routing a team's queries to the same warehouse beats spreading them around.

The next cell pulls the real numbers so you are not going on feel.

If you come back and run this notebook a second time, suspend the timing warehouse first,
otherwise run 1 starts warm and the comparison collapses:

```sql
ALTER WAREHOUSE FISERV_101_TIMING_WH SUSPEND;
```

In [ ]:
%%sql -r timings
-- Read back the actual server-side execution times for the three runs.
-- QUERY_HISTORY_BY_SESSION covers only this notebook session, so you see your
-- own runs and nobody else's. The QUALIFY keeps it to the three most recent, so
-- re-running the notebook does not stack up extra bars on the chart.
-- BYTES_SCANNED stays roughly constant across the three: the same data is read
-- every time. Only the time changes.
SELECT WAREHOUSE_NAME,
       WAREHOUSE_SIZE,
       EXECUTION_TIME AS EXECUTION_MS,
       COMPILATION_TIME AS COMPILE_MS,
       BYTES_SCANNED
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION())
WHERE QUERY_TEXT ILIKE 'SELECT COUNT(DISTINCT t.TRANSACTION_ID)%'
QUALIFY ROW_NUMBER() OVER (ORDER BY START_TIME DESC) <= 3
ORDER BY START_TIME;

In [ ]:
# Chart the three execution times so the cache effect is obvious at a glance.
import matplotlib.pyplot as plt

# SQL cell results are pandas in Legacy Notebooks but Snowpark DataFrames in Notebooks
# in Workspaces on Runtime 2.6 and above. Convert only when the method is there, so this
# cell works on either without assuming which runtime the attendee picked.
_as_pandas = getattr(timings, "to_pandas", None)
df = _as_pandas() if _as_pandas else timings.copy()

# The timings query is ordered by START_TIME and capped at three rows, so position maps
# to run: 1 cold X-Small, 2 cold Small, 3 warm X-Small.
run_labels = ["1. X-Small\ncold", "2. Small\ncold", "3. X-Small\nwarm"]
labels = run_labels[: len(df)]
values = [r.EXECUTION_MS / 1000.0 for r in df.itertuples()]
# Valencia Orange for the Small run, Snowflake Blue for the two X-Small runs.
colours = ["#FF9F36" if s == "Small" else "#29B5E8" for s in df["WAREHOUSE_SIZE"]]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, values, color=colours)
ax.set_ylabel("Execution time (seconds)")
ax.set_title("Same query three times: size changed little, the cache changed a lot")
for i, v in enumerate(values):
    ax.text(i, v, f"{v:.2f}s", ha="center", va="bottom")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Hand-off — read the query profile in Snowsight

**Why:** the numbers above tell you how long the query took. The query profile tells you *where* the time went, which is the only way to fix a slow query. It is a UI feature, so it does not run in a notebook cell.

**Steps**

1. In the left-hand nav of Snowsight, choose **Monitoring → Query History**.
2. Filter on **User** = your own user, and find the three `COUNT(DISTINCT t.TRANSACTION_ID)` queries you just ran.
3. Click the **first** one (the slowest X-Small run).
4. Open the **Query Profile** tab.
5. Look at the largest node in the tree. It will be a join. Note the **Bytes spilled to local storage** figure in the Statistics panel, if there is one.
6. In the same Statistics panel find **Percentage scanned from cache**. Compare that figure between the first X-Small run and the third one.
7. Now open the profile for the **Small** run and compare the same join node.

**What you should see:** the join is the expensive step in every run. The third X-Small run shows a much higher percentage scanned from cache than the first, which is the whole explanation for why it was faster. On the X-Small you may also see bytes spilled to local storage, meaning the warehouse ran out of memory and had to write intermediate results to disk. Spilling is the single most common reason a bigger warehouse genuinely does pay for itself.

**Docs:** https://docs.snowflake.com/en/user-guide/ui-snowsight-activity

**Return here** when you have compared the profiles.

## 3. Putting a limit on spend

Compute is the bill. A **resource monitor** is a credit budget attached to one or more warehouses: you set a quota per interval, and you say what happens when it is reached — notify, stop accepting new queries, or stop immediately.

This is the control that means an accident costs you a notification rather than a quarter.

In [ ]:
%%sql -r create_monitor
-- A 5-credit monthly budget with three escalating triggers.
-- SUSPEND lets running queries finish; SUSPEND_IMMEDIATE does not.
CREATE OR REPLACE RESOURCE MONITOR FISERV_101_MONITOR
  WITH CREDIT_QUOTA = 5
       FREQUENCY = MONTHLY
       START_TIMESTAMP = IMMEDIATELY
  TRIGGERS ON 75 PERCENT DO NOTIFY
           ON 95 PERCENT DO SUSPEND
           ON 100 PERCENT DO SUSPEND_IMMEDIATE;

In [ ]:
%%sql -r attach_monitor
-- Attach the budget to both 101 warehouses. A warehouse can have one monitor.
ALTER WAREHOUSE FISERV_101_WH SET RESOURCE_MONITOR = FISERV_101_MONITOR;

In [ ]:
%%sql -r monitor_state
-- Confirm the monitor exists and see how much of the quota you have used today.
SHOW RESOURCE MONITORS LIKE 'FISERV_101_MONITOR';

**Worth knowing:** a resource monitor is a backstop, not a cost control. It acts after the credits are spent, and it can only suspend whole warehouses. Use it to catch mistakes. Use warehouse sizing, `AUTO_SUSPEND` and sensible workload separation to actually manage cost.

Also note it did not ask you for a warehouse size, a cluster count or anything about the queries. Budget and compute are separate concerns.

## 4. Transformations

So far you have read data. Now you will write some.

The question: **which merchant categories are actually worth having, by region?** Approved volume, not attempted volume, because a declined transaction earns nothing.

Note the `TRANSACTION_RESULT = 'Approved'` filter. Getting that wrong is the most common error in payments reporting, and it overstates every number you produce.

In [ ]:
%%sql -r build_summary
-- Approved volume by region and category, written to the ANALYTICS schema.
-- CREATE OR REPLACE means you can safely re-run this cell.
CREATE OR REPLACE TABLE FISERV_101_DB.ANALYTICS.CATEGORY_BY_REGION AS
SELECT
    m.ACQUIRING_REGION,
    m.MERCHANT_CATEGORY,
    COUNT(*)                            AS APPROVED_TRANSACTIONS,
    COUNT(DISTINCT m.MERCHANT_ID)       AS ACTIVE_MERCHANTS,
    ROUND(SUM(t.AMOUNT_EUR), 2)         AS APPROVED_VOLUME_EUR,
    ROUND(AVG(t.AMOUNT_EUR), 2)         AS AVERAGE_TICKET_EUR
FROM FISERV_101_DB.RAW.TRANSACTIONS t
JOIN FISERV_101_DB.RAW.MERCHANTS m
  ON t.MERCHANT_ID = m.MERCHANT_ID
WHERE t.TRANSACTION_RESULT = 'Approved'
GROUP BY m.ACQUIRING_REGION, m.MERCHANT_CATEGORY;

In [ ]:
%%sql -r top_categories
-- The top three categories by approved volume within each region.
-- QUALIFY filters on a window function without needing a subquery.
SELECT ACQUIRING_REGION, MERCHANT_CATEGORY, APPROVED_VOLUME_EUR, AVERAGE_TICKET_EUR
FROM FISERV_101_DB.ANALYTICS.CATEGORY_BY_REGION
QUALIFY ROW_NUMBER() OVER (PARTITION BY ACQUIRING_REGION
                           ORDER BY APPROVED_VOLUME_EUR DESC) <= 3
ORDER BY ACQUIRING_REGION, APPROVED_VOLUME_EUR DESC;

In [ ]:
# Grouped bar chart of the top categories by region.
import matplotlib.pyplot as plt

# SQL cell results are pandas in Legacy Notebooks but Snowpark DataFrames in Notebooks
# in Workspaces on Runtime 2.6 and above. Convert only when the method is there, so this
# cell works on either without assuming which runtime the attendee picked.
_as_pandas = getattr(top_categories, "to_pandas", None)
df = _as_pandas() if _as_pandas else top_categories.copy()
df["VOL_M"] = df["APPROVED_VOLUME_EUR"] / 1_000_000.0
pivot = df.pivot(index="ACQUIRING_REGION", columns="MERCHANT_CATEGORY", values="VOL_M")

ax = pivot.plot(kind="bar", figsize=(9, 4.5),
                color=["#29B5E8", "#11567F", "#71D3DC", "#FF9F36", "#7D44CF", "#D45B90"])
ax.set_ylabel("Approved volume (\u20ac millions)")
ax.set_xlabel("")
ax.set_title("Top merchant categories by acquiring region")
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Zero-copy cloning and Time Travel

You now have a table other people might depend on. Before you change it, take a copy.

A **zero-copy clone** creates a new table that shares the original's storage. It is instant regardless of table size and costs nothing until one of the two diverges. **Time Travel** lets you query a table as it was at a point in the past, within its retention window.

Together they replace most of what a backup schedule used to do.

In [ ]:
%%sql -r clone_table
-- Instant copy. No data is moved, so the size of the table is irrelevant.
CREATE OR REPLACE TABLE FISERV_101_DB.ANALYTICS.CATEGORY_BY_REGION_BACKUP
  CLONE FISERV_101_DB.ANALYTICS.CATEGORY_BY_REGION;

In [ ]:
%%sql -r break_it
-- Now break the original, the way a bad WHERE clause in a script would.
-- The second statement remembers the query id, so the next cell can ask for the
-- table as it was immediately before this deletion.
DELETE FROM FISERV_101_DB.ANALYTICS.CATEGORY_BY_REGION
WHERE ACQUIRING_REGION = 'DACH';
SET delete_qid = LAST_QUERY_ID();

In [ ]:
%%sql -r recover
-- Three views of the same table: as it is now, as it was before the DELETE,
-- and the clone you took beforehand. Two of the three still have DACH.
SELECT 'now' AS VERSION, COUNT(*) AS ROW_COUNT
FROM FISERV_101_DB.ANALYTICS.CATEGORY_BY_REGION
UNION ALL
SELECT 'before the delete', COUNT(*)
FROM FISERV_101_DB.ANALYTICS.CATEGORY_BY_REGION BEFORE (STATEMENT => $delete_qid)
UNION ALL
SELECT 'the clone', COUNT(*)
FROM FISERV_101_DB.ANALYTICS.CATEGORY_BY_REGION_BACKUP;

In [ ]:
%%sql -r restore
-- Put the deleted rows back from Time Travel. No backup file, no restore job.
INSERT INTO FISERV_101_DB.ANALYTICS.CATEGORY_BY_REGION
  (ACQUIRING_REGION, MERCHANT_CATEGORY, APPROVED_TRANSACTIONS,
   ACTIVE_MERCHANTS, APPROVED_VOLUME_EUR, AVERAGE_TICKET_EUR)
SELECT ACQUIRING_REGION, MERCHANT_CATEGORY, APPROVED_TRANSACTIONS,
       ACTIVE_MERCHANTS, APPROVED_VOLUME_EUR, AVERAGE_TICKET_EUR
FROM FISERV_101_DB.ANALYTICS.CATEGORY_BY_REGION_BACKUP
WHERE ACQUIRING_REGION = 'DACH';

## Hand-off — find something you do not know the name of

**Why:** everything so far assumed you knew the table name. In a real account with thousands of objects you usually do not. **Universal Search** searches object names, column names, descriptions and even Marketplace listings from one box, and it is a UI feature.

**Steps**

1. Click **Search** at the top of the Snowsight left-hand nav (or press `/`).
2. Type `acquiring region`. You did not create anything with that name, but you created a *column* called `ACQUIRING_REGION`.
3. Look at the results. Note that it found tables by their column names, and note the **Data products** and **Documentation** sections underneath.
4. Now search `chargeback`. Nothing in your 101 database uses that word.
5. Look at what came back from the day-two database and from the Snowflake documentation.

**What you should see:** `CATEGORY_BY_REGION`, `MERCHANTS` and `TRANSACTIONS` for the first search, matched on column name rather than table name. For the second, results from `FISERV_PAYMENTS_DB` — the database you will use tomorrow — plus documentation hits.

**Docs:** https://docs.snowflake.com/en/user-guide/ui-snowsight-universal-search

**Return here** when you have run both searches.

## What you did

- Ran the same query three times and found that the **local cache mattered more than the warehouse size**
- Read a query profile and looked for spilling, which is when size genuinely does help
- Attached a **resource monitor** as a backstop against runaway spend
- Built a transformation into `ANALYTICS.CATEGORY_BY_REGION`, filtering to approved transactions only
- Broke that table and recovered it two different ways, with a **zero-copy clone** and with **Time Travel**
- Found objects by column name using **Universal Search**

## Where this goes

Everything here was structured data you were handed as a table. In **Part 2** the data arrives as JSON files on a stage, which is how most of it actually turns up, and you will build a pipeline that keeps a transformation up to date on its own instead of you re-running a `CREATE OR REPLACE`.

Leave this notebook open. Part 2 uses the same database.